<a href="https://colab.research.google.com/github/kavipriya30/Kavipriya-Codeboosters-internship-2026/blob/main/DAY_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
print("All libraries imported successfully!")
print(f"pandas :{pd.__version__}")
print(f"requests:{requests.__version__}")

All libraries imported successfully!
pandas :2.2.2
requests:2.32.4


In [15]:
raw_df = pd.read_csv('messy_sales_data.csv')
print(f"Dataset loaded: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print(f"Columns: {raw_df.columns.tolist()}")
print("\nFirst 3 rows:")
print(raw_df.head(3))

Dataset loaded: 30 rows, 9 columns
Columns: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 3 rows:
   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  


In [16]:
print('='*55)
print(' DATA QUALITY DIAGNOSIS REPORT')
print('='*55)
print('\n[1] MISSING VALUES per column:')
print(raw_df.isnull().sum())
print(f'\n[2] DUPLICATE ROWS:{raw_df.duplicated().sum()}')
print('\n[3] DATA TYPES:')
print(raw_df.dtypes)
print('\n[4] UNIQUE CATAGORIES:',raw_df['category'].unique())
print('[4] Sample customer name:',raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_date values:',raw_df['order_date'].unique()[:6])

 DATA QUALITY DIAGNOSIS REPORT

[1] MISSING VALUES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS:0

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATAGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer name: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [17]:
df = raw_df.copy()
print(f"Working copy created: {df.shape}")
print("raw_df is untouched - we can always reset by running df = raw_df.copy()")

Working copy created: (30, 9)
raw_df is untouched - we can always reset by running df = raw_df.copy()


In [19]:
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f"Filled missing quantity with median: {median_qty}")
df['category'].fillna('Uncategorized', inplace=True)
print("After fixing nulls:", df.isnull().sum().sum(), "total missing values")

Filled missing quantity with median: 2.0
After fixing nulls: 3 total missing values


/tmp/ipykernel_4459/1753537980.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['quantity'].fillna(median_qty, inplace=True)


In [20]:
print(f"Before duplication removal: {len(df)} rows")

print(f"Duplicate rows: {df.duplicated().sum()}")

print("\nDuplicate rows:")
print(df[df.duplicated(keep=False)][["order_id", "customer_name", "product", "order_date"]])

df.drop_duplicates(inplace=True)

print(f"After duplication removal: {len(df)} rows")

print(f"Rows removed: {len(raw_df) - len(df)}")

Before duplication removal: 30 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
After duplication removal: 30 rows
Rows removed: 0


In [21]:
print("Sample dates before parsing:")
print(df["order_date"].head().tolist())

df["order_date"] = pd.to_datetime(
    df["order_date"],
    dayfirst=False,
    errors="coerce"
)

Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05']


In [22]:
nat_count = df["order_date"].isnull().sum()

print(f"\nUnparsable dates (NaT): {nat_count}")

df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.strftime("%B")

print("\nSample dates after parsing:")
print(df[["order_date", "year", "month", "month_name"]].head(5))


Unparsable dates (NaT): 2

Sample dates after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [23]:
print("Before standardlization:",df["customer_name"].unique()[:6])
df["custoomer_name"] = (
df["customer_name"]
.str.title()
.str.strip()
)
print("After standardlization:",df["customer_name"].unique()[:6])
print(f"\nBefore:keyboard rows with Electronics category:")
wrong_mask=(df["product"]=="keyboard")&(df["category"]=="Electronics")
print(df[wrong_mask][["product","category"]])
df.loc[wrong_mask,"category"]="Accessories"
print("After fix:unique categories:",df["category"].unique())

Before standardlization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After standardlization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']

Before:keyboard rows with Electronics category:
Empty DataFrame
Columns: [product, category]
Index: []
After fix:unique categories: ['Electronics' 'Accessories' 'uncategorized']


In [29]:
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").astype(int)

df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

df["revenue"] = df["quantity"] * df["unit_price"]

print("Revenue column created:")

print(df[["customer_name", "product", "quantity", "unit_price", "revenue"]].head(5))

print(f"\nTotal revenue across all orders: {df['revenue'].sum():,.2f}")

Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop         2       45000    90000
1    Priya Nair       NaN         1       15000    15000
2    AMIT VERMA  Keyboard         3        1200     3600
3  Sunita Patel   Monitor         0       22000        0
4  Ramesh Kumar    Laptop         2       45000    90000

Total revenue across all orders: 679,000.00


In [30]:
# Create revenue column if not already created
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").fillna(0).astype(int)

df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce").fillna(0)

df["revenue"] = df["quantity"] * df["unit_price"]

# Validation Report
print('=' * 55)
print('POST-CLEANING VALIDATION REPORT')
print('=' * 55)

print(f'Original rows  : {len(raw_df)}')
print(f'Cleaned rows   : {len(df)}')
print(f'Rows removed   : {len(raw_df) - len(df)} (duplicates)')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicate      : {df.duplicated().sum()}')
print(f'Date nulls     : {df["order_date"].isnull().sum()}')
print(f'Revenue NaN    : {df["revenue"].isnull().sum()}')
print(f'Categories     : {sorted(df["category"].dropna().astype(str).unique())}')

all_clean = (
    df.isnull().sum().sum() == 0 and
    df.duplicated().sum() == 0
)

print(f'DATA IS CLEAN: {all_clean}')

POST-CLEANING VALIDATION REPORT
Original rows  : 30
Cleaned rows   : 30
Rows removed   : 0 (duplicates)
Missing values : 4
Duplicate      : 0
Date nulls     : 0
Revenue NaN    : 0
Categories     : ['Accessories', 'Electronics']
DATA IS CLEAN: False


In [32]:
product_rev = (
    df.groupby("product")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Revenue by Product:")
print(product_rev.to_string(index=False))

category_summary = (
    df.groupby("category")
    .agg(
        total_revenue=("revenue", "sum"),
        avg_order_value=("revenue", "mean"),
        avg_orders=("order_id", "count"),
        unique_products=("product", "nunique")
    )
    .round(2)
    .reset_index()
)

print("\nCategory Summary:")
print(category_summary.to_string(index=False))

Revenue by Product:
product
450000
110000
 28000
 20800
 20400
 19800
 15000

Category Summary:
   category  total_revenue  avg_order_value  avg_orders  unique_products
Accessories          71200          5476.92          13                4
Electronics         563800         35237.50          16                4


In [33]:
df.to_csv("cleaned_sales_data.csv", index=False)
print("Cleaned data saved to cleaned_sales_data.csv")
print(f"Final dataset:{df.shape[0]}rows={df.shape[1]}columns")
print("\nETL Pipeline for Sales Date:COMPLETE")
print("EXTRACT->messy_sales_data.csv")
print("TRANSFORM->cleaned_sales_data.csv")
print("LOAD->cleaned_sales_data.csv")

Cleaned data saved to cleaned_sales_data.csv
Final dataset:30rows=10columns

ETL Pipeline for Sales Date:COMPLETE
EXTRACT->messy_sales_data.csv
TRANSFORM->cleaned_sales_data.csv
LOAD->cleaned_sales_data.csv


In [35]:
API_KEY = "3344566facc2742e7ff62deefe409ef0"
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
CITIES = [
    "Mumbai",
    "Delhi",
    "Bangalore",
    "Chennai",
    "Hyderabad",
    "Kolkata",
    "Pune",
    "Jaipur"
]
print(f"API configured for {len(CITIES)} cities")
print(f"Cities: {CITIES}")
print("\nBase URL:", BASE_URL)

API configured for 8 cities
Cities: ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

Base URL: https://api.openweathermap.org/data/2.5/weather


In [37]:
import pandas as pd
import numpy as np

# Sample sales data
data = {
    "customer_name": ["Amit", "Sara", "Amit", "John", None],
    "product": ["Laptop", "Mouse", "Laptop", "Keyboard", "Mouse"],
    "price": [50000, 500, 50000, np.nan, 700],
    "quantity": [1, 2, 1, 1, np.nan]
}

df = pd.DataFrame(data)

print(df)

  customer_name   product    price  quantity
0          Amit    Laptop  50000.0       1.0
1          Sara     Mouse    500.0       2.0
2          Amit    Laptop  50000.0       1.0
3          John  Keyboard      NaN       1.0
4          None     Mouse    700.0       NaN


Q1 What are the three stages of ETL?


In [53]:

# EXTRACT
print("Original Data")
print(df)

# TRANSFORM
df_clean = df.dropna()

# LOAD
df_clean.to_csv("cleaned_sales.csv", index=False)

print("\nCleaned Data")
print(df_clean)

Original Data
  customer_name   product    price  quantity
0          Amit    Laptop  50000.0       1.0
1          Sara     Mouse    500.0       2.0
2          Amit    Laptop  50000.0       1.0
3          John  Keyboard  25350.0       1.0
4          None     Mouse    700.0       NaN

Cleaned Data
  customer_name   product    price  quantity
0          Amit    Laptop  50000.0       1.0
1          Sara     Mouse    500.0       2.0
2          Amit    Laptop  50000.0       1.0
3          John  Keyboard  25350.0       1.0


Q2: DataFrame rows reduced from 500 → 412 after dropna()

In [48]:


original_rows = len(df)

cleaned_df = df.dropna()

new_rows = len(cleaned_df)

removed = original_rows - new_rows

print("Rows removed:", removed)

Rows removed: 1


Q3: Remove duplicates based on customer_name AND product

In [49]:

df_no_duplicates = df.drop_duplicates(
    subset=["customer_name", "product"]
)

print(df_no_duplicates)

  customer_name   product    price  quantity
0          Amit    Laptop  50000.0       1.0
1          Sara     Mouse    500.0       2.0
3          John  Keyboard  25350.0       1.0
4          None     Mouse    700.0       NaN


Q4: Difference between fillna(0) and fillna(df['col'].median())

In [50]:

df_zero = df.fillna(0)

print(df_zero)

  customer_name   product    price  quantity
0          Amit    Laptop  50000.0       1.0
1          Sara     Mouse    500.0       2.0
2          Amit    Laptop  50000.0       1.0
3          John  Keyboard  25350.0       1.0
4             0     Mouse    700.0       0.0


Q5: Python code to call weather API for Delhi

In [51]:

median_price = df["price"].median()

df["price"] = df["price"].fillna(median_price)

print(df)

  customer_name   product    price  quantity
0          Amit    Laptop  50000.0       1.0
1          Sara     Mouse    500.0       2.0
2          Amit    Laptop  50000.0       1.0
3          John  Keyboard  25350.0       1.0
4          None     Mouse    700.0       NaN


In [43]:
import requests

Q6: Meaning of response.status_code == 200 and 401

In [52]:

api_key = "3344566facc2742e7ff62deefe409ef0"
city = "Delhi"

url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

response = requests.get(url)

print(response.status_code)

data = response.json()

print(data)

200
{'coord': {'lon': 77.2167, 'lat': 28.6667}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'base': 'stations', 'main': {'temp': 44.05, 'feels_like': 41.11, 'temp_min': 44.05, 'temp_max': 44.05, 'pressure': 999, 'humidity': 10, 'sea_level': 999, 'grnd_level': 974}, 'visibility': 7000, 'wind': {'speed': 5.14, 'deg': 280, 'gust': 10.29}, 'clouds': {'all': 0}, 'dt': 1779877402, 'sys': {'type': 1, 'id': 9165, 'country': 'IN', 'sunrise': 1779839709, 'sunset': 1779889301}, 'timezone': 19800, 'id': 1273294, 'name': 'Delhi', 'cod': 200}


In [46]:
if response.status_code == 200:
    print("Success")
elif response.status_code == 401:
    print("Unauthorized - Check API key")
else:
    print("Error")

Success
